# TrustMind AI — Knowledge Base Collection

**MSc Artificial Intelligence Dissertation**

## Research question

> To what extent does Retrieval-Augmented Generation (RAG) improve the trustworthiness, reliability, and explainability of LLM-generated wellbeing assessments compared with standalone LLMs?

## Purpose of this notebook

This notebook documents the **trusted-source collection** stage for TrustMind AI.

It downloads **only manually approved URLs**, extracts readable article text, stores raw HTML + cleaned Markdown, and records an auditable manifest. Every document remains `review_status=pending` until you approve it by hand.

**Not implemented here:** chunking, embeddings, FAISS, BM25, hybrid retrieval, reranking, or LLM+RAG inference.


## Section 1 — SWMH vs knowledge base

| Resource | Role in the dissertation |
|----------|--------------------------|
| **SWMH dataset** | Evaluation benchmark for LLM-only vs LLM+RAG experiments (Reddit posts + subreddit labels) |
| **Knowledge base (this stage)** | Curated NHS / Mind / Samaritans / UWE-style guidance that a future RAG system may retrieve as evidence |

Keeping these separate preserves experimental clarity: SWMH measures task performance; the knowledge base supplies **external grounding** for trustworthiness and explainability.


## Section 2 — Imports and paths


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "research":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "scripts"))

KB = ROOT / "knowledge_base"
SOURCES_CSV = KB / "sources" / "approved_sources.csv"
MANIFEST_CSV = KB / "metadata" / "source_manifest.csv"
CLEANED_DIR = KB / "cleaned"

print("Project root:", ROOT)
print("Knowledge base:", KB)


## Section 3 — Approved source list

Only rows with `approved_for_collection=true` are downloaded. New URLs must be added manually — the collector does not crawl.


In [ ]:
from collect_sources import load_approved_sources

all_sources = pd.read_csv(SOURCES_CSV)
collectible = load_approved_sources(SOURCES_CSV)

print("All listed sources:", len(all_sources))
print("Approved for collection:", len(collectible))
display(all_sources)


## Section 4 — Run the source collector

This calls `scripts/collect_sources.py` programmatically (same logic as the CLI).

Set `FORCE = True` only if you intentionally want to re-download existing sources.


In [ ]:
from collect_sources import collect_all_approved_sources

FORCE = False  # set True to re-fetch
manifest = collect_all_approved_sources(force=FORCE)
print("\nManifest rows:", len(manifest))
display(manifest)


## Section 5 — Collection results and coverage


In [ ]:
if MANIFEST_CSV.exists():
    manifest = pd.read_csv(MANIFEST_CSV)
else:
    manifest = pd.DataFrame()

print("Collection status counts:")
print(manifest.get("collection_status", pd.Series(dtype=str)).value_counts())
print("\nReview status counts:")
print(manifest.get("review_status", pd.Series(dtype=str)).value_counts())

if not manifest.empty:
    print("\nDocuments by organisation:")
    display(manifest.groupby("organisation").size().rename("count").reset_index())
    print("Documents by topic:")
    display(manifest.groupby("topic").size().rename("count").reset_index())

    success = manifest[manifest["collection_status"] == "success"]
    if not success.empty:
        print("\nWord-count statistics (successful collections):")
        display(success["word_count"].describe())


## Section 6 — Manual review: NHS depression overview

Inspect the cleaned Markdown. Confirm headings, support information, and any emergency guidance look intact before approving.


In [ ]:
nhs_path = CLEANED_DIR / "NHS_DEP_001.md"
if nhs_path.exists():
    text = nhs_path.read_text(encoding="utf-8")
    print(f"File: {nhs_path} ({len(text.split())} whitespace-separated tokens in full file)\n")
    print(text[:4000])
    if len(text) > 4000:
        print("\n... [truncated for notebook display — open the file for full text] ...")
else:
    print("NHS_DEP_001.md not found. Run the collector first.")


## Section 7 — Validation checks


In [ ]:
from validate_knowledge_base import run_validation, REPORT_PATH

report = run_validation()
print("\nValidation report path:", REPORT_PATH)


## Section 8 — How to approve a document (manual)

1. Read `knowledge_base/cleaned/SOURCE_ID.md`.
2. If acceptable, copy it to `knowledge_base/review/approved/`.
3. Set `review_status: approved` in the YAML front matter **and** in `source_manifest.csv`.
4. If not acceptable, place it under `knowledge_base/review/rejected/` with `review_status: rejected`.

The future RAG index must use **approved** documents only.


## Section 9 — Stage boundary

At the end of this notebook:

- Trusted pages can be collected reproducibly with full provenance.
- Documents await human review (`pending`).
- **Retrieval and RAG have not been implemented.**

Next experimental stage (later): chunk approved Markdown → embed → FAISS/BM25 → hybrid retrieve → GPT-4.1 with citations → compare to the LLM-only baseline.

---

*End of knowledge-base collection notebook.*
